# LightGBM Model Training on ZB Stock Data

This notebook trains a LightGBM model on ZB (Shanghai & Shenzhen main board) stock market data and evaluates performance on validation and test datasets.

## Workflow
1. Load and explore ZB training/validation data
2. Preprocess and engineer features
3. Train LightGBM model
4. Evaluate on validation set
5. Load test data and make predictions
6. Calculate comprehensive performance metrics

## Section 1: Import Required Libraries ✅

Import LightGBM, pandas, numpy, scikit-learn, and visualization libraries for model training and evaluation.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# LightGBM and ML libraries
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve
)

import warnings
warnings.filterwarnings("ignore")

# Set up paths and plotting
workspace_root = Path("/Users/alex/rust_llm_stock")
if Path.cwd() != workspace_root:
    os.chdir(workspace_root)

sns.set(style='whitegrid', context='notebook', rc={'figure.figsize': (12, 6)})
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"✅ Libraries loaded successfully")
print(f"   LightGBM: {lgb.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")

: 

## Section 2: Load and Explore ZB Training Data 📊

Load the ZB training and validation datasets, display basic information, and check for missing values.

In [ ]:
# Load ZB training and validation data
train_path = Path("./data/zb_train.csv")
val_path = Path("./data/zb_val.csv")

if not train_path.exists():
    print(f"⚠️  {train_path} not found. Check if export_training_data was run.")
else:
    df_train = pd.read_csv(train_path)
    print(f"✅ Loaded training data: {df_train.shape}")
    print(f"   Columns: {df_train.shape[1]}")
    print(f"   Rows: {df_train.shape[0]:,}")
    
if not val_path.exists():
    print(f"⚠️  {val_path} not found.")
else:
    df_val = pd.read_csv(val_path)
    print(f"✅ Loaded validation data: {df_val.shape}")
    print(f"   Rows: {df_val.shape[0]:,}")

# Display sample data
print(f"\n📋 Sample training data (first 3 rows):")
display(df_train.iloc[:3, :10])

# Data types and missing values
print(f"\n📊 Data types summary:")
print(f"   Numeric: {(df_train.dtypes == 'float64').sum() + (df_train.dtypes == 'int64').sum()}")
print(f"   Object: {(df_train.dtypes == 'object').sum()}")

print(f"\n❌ Missing values in training set:")
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(2)
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
if len(missing_df) > 0:
    display(missing_df.head(10))
else:
    print("   ✅ No missing values")

# Target variable exploration
print(f"\n🎯 Target variable distribution (next_day_direction):")
target = 'next_day_direction'
if target in df_train.columns:
    dist = df_train[target].value_counts(dropna=False).sort_index()
    dist_pct = (dist / len(df_train) * 100).round(2)
    target_summary = pd.DataFrame({'Count': dist, 'Percentage': dist_pct})
    display(target_summary)
    
    # Check balance
    if len(dist) >= 2:
        balance_ratio = dist.min() / dist.max()
        print(f"   Balance ratio: {balance_ratio:.3f}")
        if balance_ratio < 0.3:
            print(f"   ⚠️  IMBALANCED - Consider class_weight in LightGBM")
        elif balance_ratio < 0.5:
            print(f"   ⚙️  Moderately imbalanced")
        else:
            print(f"   ✅ Well balanced")
else:
    print(f"   ⚠️  Target column '{target}' not found")

## Section 3: Data Preprocessing and Feature Engineering 🔧

Clean data, handle missing values, encode categorical features, and prepare data for LightGBM.

In [ ]:
# Identify columns
target_col = 'next_day_direction'
id_cols = ['ts_code', 'trade_date']
exclude_cols = {'ts_code', 'trade_date', 'next_day_direction', 'next_3day_direction', 
                'next_day_return', 'next_3day_return', 'id', 'created_at'}
exclude_prefixes = ('industry_emb_', 'act_ent_type_emb_')

# Get numeric and categorical features
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_train.select_dtypes(include=['object']).columns.tolist()

# Filter out excluded columns
feature_cols = [c for c in numeric_cols if c not in exclude_cols and not any(c.startswith(p) for p in exclude_prefixes)]
categorical_feature_cols = [c for c in categorical_cols if c not in exclude_cols]

# Include embeddings as features
embedding_cols = [c for c in numeric_cols if any(c.startswith(p) for p in exclude_prefixes)]

all_feature_cols = feature_cols + categorical_feature_cols + embedding_cols

print(f"📋 Feature breakdown:")
print(f"   Numeric features: {len(feature_cols)}")
print(f"   Categorical features: {len(categorical_feature_cols)}")
print(f"   Embedding features: {len(embedding_cols)}")
print(f"   Total features: {len(all_feature_cols)}")

# Handle missing values in numeric features
print(f"\n🔧 Handling missing values...")
imputer = SimpleImputer(strategy='median')
df_train[feature_cols] = imputer.fit_transform(df_train[feature_cols])
df_val[feature_cols] = imputer.transform(df_val[feature_cols])

# Handle missing values in embeddings
for emb_col in embedding_cols:
    df_train[emb_col].fillna(0.0, inplace=True)
    df_val[emb_col].fillna(0.0, inplace=True)

# Encode categorical features
print(f"🔧 Encoding categorical features...")
label_encoders = {}
for cat_col in categorical_feature_cols:
    le = LabelEncoder()
    df_train[cat_col] = le.fit_transform(df_train[cat_col].astype(str))
    df_val[cat_col] = le.transform(df_val[cat_col].astype(str))
    label_encoders[cat_col] = le
    print(f"   {cat_col}: {len(le.classes_)} unique values")

# Extract features and target
X_train = df_train[all_feature_cols]
y_train = df_train[target_col]

X_val = df_val[all_feature_cols]
y_val = df_val[target_col]

print(f"\n✅ Preprocessing complete:")
print(f"   Training set: {X_train.shape}")
print(f"   Validation set: {X_val.shape}")
print(f"   Target distribution (train): {y_train.value_counts().to_dict()}")
print(f"   Target distribution (val): {y_val.value_counts().to_dict()}")

## Section 4: Train LightGBM Model 🚀

Create and train LightGBM model with optimized hyperparameters on training set and evaluate on validation set.

In [ ]:
# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': RANDOM_SEED,
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'min_data_in_leaf': 20,
    'is_unbalance': True if (y_train.value_counts().min() / y_train.value_counts().max()) < 0.3 else False,
}

print("🚀 Training LightGBM model...")
print(f"   Parameters: {params}")

# Train the model
model = lgb.train(
    params,
    train_data,
    num_boost_round=200,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.log_evaluation(period=20),
        lgb.early_stopping(stopping_rounds=20)
    ]
)

print(f"\n✅ Model training complete!")
print(f"   Best iteration: {model.best_iteration}")
print(f"   Best training AUC: {model.best_score['train']['auc']:.4f}")
print(f"   Best validation AUC: {model.best_score['valid']['auc']:.4f}")

# Make predictions on validation set
y_pred_val_proba = model.predict(X_val)
y_pred_val = (y_pred_val_proba >= 0.5).astype(int)

# Calculate validation metrics
val_accuracy = accuracy_score(y_val, y_pred_val)
val_precision = precision_score(y_val, y_pred_val, zero_division=0)
val_recall = recall_score(y_val, y_pred_val, zero_division=0)
val_f1 = f1_score(y_val, y_pred_val, zero_division=0)
val_auc = roc_auc_score(y_val, y_pred_val_proba)

print(f"\n📊 Validation Set Performance:")
print(f"   Accuracy:  {val_accuracy:.4f}")
print(f"   Precision: {val_precision:.4f}")
print(f"   Recall:    {val_recall:.4f}")
print(f"   F1-Score:  {val_f1:.4f}")
print(f"   AUC-ROC:   {val_auc:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': all_feature_cols,
    'Importance': model.feature_importance(importance_type='gain')
}).sort_values('Importance', ascending=False)

print(f"\n🎯 Top 15 Most Important Features:")
display(feature_importance.head(15))

# Visualize feature importance
fig, ax = plt.subplots(figsize=(12, 6))
top_features = feature_importance.head(20)
ax.barh(range(len(top_features)), top_features['Importance'].values)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top 20 Feature Importance - LightGBM ZB Model')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('./artifacts/feature_importance_lgb.png', dpi=100, bbox_inches='tight')
plt.show()

## Section 5: Load Test Data 📁

Load the test dataset from ./data/test_data.csv and apply the same preprocessing steps used on training data.

In [ ]:
# Load test data
test_path = Path("./data/test_data.csv")

if not test_path.exists():
    print(f"⚠️  {test_path} not found.")
else:
    df_test = pd.read_csv(test_path)
    print(f"✅ Loaded test data: {df_test.shape}")
    print(f"   Rows: {df_test.shape[0]:,}")
    print(f"   Columns: {df_test.shape[1]}")
    
    # Display sample test data
    print(f"\n📋 Sample test data (first 3 rows):")
    display(df_test.iloc[:3, :10])
    
    # Check for target column in test data
    if target_col in df_test.columns:
        print(f"\n🎯 Test set target distribution:")
        test_dist = df_test[target_col].value_counts(dropna=False)
        print(f"   {test_dist.to_dict()}")
        has_test_target = True
    else:
        print(f"\n⚠️  Target column '{target_col}' not in test data")
        has_test_target = False
    
    # Apply preprocessing to test data
    print(f"\n🔧 Preprocessing test data...")
    
    # Handle missing values in numeric features
    df_test[feature_cols] = imputer.transform(df_test[feature_cols])
    
    # Handle missing values in embeddings
    for emb_col in embedding_cols:
        df_test[emb_col].fillna(0.0, inplace=True)
    
    # Encode categorical features using fitted encoders
    for cat_col in categorical_feature_cols:
        if cat_col in label_encoders:
            df_test[cat_col] = label_encoders[cat_col].transform(df_test[cat_col].astype(str))
    
    # Extract features
    X_test = df_test[all_feature_cols]
    
    # Extract target if available
    if has_test_target:
        y_test = df_test[target_col]
    else:
        y_test = None
    
    print(f"✅ Test data preprocessing complete: {X_test.shape}")
    print(f"   Ready for prediction")

## Section 6: Make Predictions on Test Data 🔮

Use the trained LightGBM model to generate predictions on the test dataset.

In [ ]:
# Make predictions on test data
print("🔮 Generating predictions on test data...")
y_pred_test_proba = model.predict(X_test)
y_pred_test = (y_pred_test_proba >= 0.5).astype(int)

print(f"✅ Predictions generated: {len(y_pred_test)} samples")
print(f"   Predicted class 0: {(y_pred_test == 0).sum()} ({(y_pred_test == 0).sum() / len(y_pred_test) * 100:.1f}%)")
print(f"   Predicted class 1: {(y_pred_test == 1).sum()} ({(y_pred_test == 1).sum() / len(y_pred_test) * 100:.1f}%)")

print(f"\n📊 Prediction probabilities:")
print(f"   Min: {y_pred_test_proba.min():.4f}")
print(f"   Max: {y_pred_test_proba.max():.4f}")
print(f"   Mean: {y_pred_test_proba.mean():.4f}")
print(f"   Median: {np.median(y_pred_test_proba):.4f}")

# Add predictions to test dataframe
df_test['predicted_proba'] = y_pred_test_proba
df_test['predicted_class'] = y_pred_test

# Save predictions
output_path = Path("./artifacts/test_predictions_lgb.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_test[['ts_code', 'trade_date'] + list(df_test.columns[-2:])].to_csv(output_path, index=False)
print(f"\n💾 Predictions saved to {output_path}")

# Display sample predictions
print(f"\n📋 Sample predictions:")
sample_pred = df_test[['ts_code', 'trade_date', 'predicted_proba', 'predicted_class']].head(10)
display(sample_pred)

## Section 7: Evaluate Model Performance 📈

Calculate comprehensive performance metrics and visualizations for test predictions.

In [ ]:
if has_test_target:
    # Calculate test metrics
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_auc = roc_auc_score(y_test, y_pred_test_proba)
    
    print("="*70)
    print("📊 TEST SET PERFORMANCE SUMMARY")
    print("="*70)
    
    print(f"\n✅ Classification Metrics:")
    print(f"   Accuracy:  {test_accuracy:.4f}")
    print(f"   Precision: {test_precision:.4f}")
    print(f"   Recall:    {test_recall:.4f}")
    print(f"   F1-Score:  {test_f1:.4f}")
    print(f"   AUC-ROC:   {test_auc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_test)
    print(f"\n📋 Confusion Matrix:")
    print(f"   TN: {cm[0,0]:6d}   FP: {cm[0,1]:6d}")
    print(f"   FN: {cm[1,0]:6d}   TP: {cm[1,1]:6d}")
    
    # Classification Report
    print(f"\n📝 Detailed Classification Report:")
    print(classification_report(y_test, y_pred_test, target_names=['Class 0', 'Class 1']))
    
    # Comparison: Training vs Validation vs Test
    print("\n" + "="*70)
    print("📊 PERFORMANCE COMPARISON: TRAIN vs VALIDATION vs TEST")
    print("="*70)
    
    train_accuracy = accuracy_score(y_train, (model.predict(X_train) >= 0.5).astype(int))
    train_auc = roc_auc_score(y_train, model.predict(X_train))
    
    comparison_df = pd.DataFrame({
        'Set': ['Training', 'Validation', 'Test'],
        'Accuracy': [train_accuracy, val_accuracy, test_accuracy],
        'Precision': [precision_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0), 
                      val_precision, test_precision],
        'Recall': [recall_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0), 
                   val_recall, test_recall],
        'F1-Score': [f1_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0), 
                     val_f1, test_f1],
        'AUC-ROC': [train_auc, val_auc, test_auc]
    })
    
    display(comparison_df.round(4))
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Confusion Matrix Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=False)
    axes[0, 0].set_title('Confusion Matrix - Test Set')
    axes[0, 0].set_ylabel('True Label')
    axes[0, 0].set_xlabel('Predicted Label')
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_test_proba)
    axes[0, 1].plot(fpr, tpr, label=f'AUC = {test_auc:.4f}', linewidth=2)
    axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    axes[0, 1].set_xlabel('False Positive Rate')
    axes[0, 1].set_ylabel('True Positive Rate')
    axes[0, 1].set_title('ROC Curve - Test Set')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # 3. Prediction Probability Distribution
    axes[1, 0].hist(y_pred_test_proba[y_test == 0], bins=30, alpha=0.6, label='Class 0', color='blue')
    axes[1, 0].hist(y_pred_test_proba[y_test == 1], bins=30, alpha=0.6, label='Class 1', color='orange')
    axes[1, 0].axvline(0.5, color='red', linestyle='--', label='Decision Threshold')
    axes[1, 0].set_xlabel('Predicted Probability')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Prediction Probability Distribution - Test Set')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    
    # 4. Metrics Comparison
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    train_metrics = [train_accuracy, 
                    precision_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0),
                    recall_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0),
                    f1_score(y_train, (model.predict(X_train) >= 0.5).astype(int), zero_division=0),
                    train_auc]
    val_metrics = [val_accuracy, val_precision, val_recall, val_f1, val_auc]
    test_metrics = [test_accuracy, test_precision, test_recall, test_f1, test_auc]
    
    x = np.arange(len(metrics))
    width = 0.25
    
    axes[1, 1].bar(x - width, train_metrics, width, label='Training', alpha=0.8)
    axes[1, 1].bar(x, val_metrics, width, label='Validation', alpha=0.8)
    axes[1, 1].bar(x + width, test_metrics, width, label='Test', alpha=0.8)
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('Performance Metrics Comparison')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(metrics, rotation=45, ha='right')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3, axis='y')
    axes[1, 1].set_ylim([0, 1.05])
    
    plt.tight_layout()
    plt.savefig('./artifacts/test_evaluation_lgb.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"\n💾 Evaluation plot saved to ./artifacts/test_evaluation_lgb.png")
    
else:
    print("⚠️  Test data does not have target values. Showing prediction distribution only.")
    print(f"\n📊 Prediction Summary:")
    print(f"   Total predictions: {len(y_pred_test)}")
    print(f"   Class 0 predictions: {(y_pred_test == 0).sum()} ({(y_pred_test == 0).sum() / len(y_pred_test) * 100:.1f}%)")
    print(f"   Class 1 predictions: {(y_pred_test == 1).sum()} ({(y_pred_test == 1).sum() / len(y_pred_test) * 100:.1f}%)")
    
    # Visualize prediction distribution
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(y_pred_test_proba, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Frequency')
    ax.set_title('Prediction Probability Distribution - Test Set')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('./artifacts/test_predictions_distribution_lgb.png', dpi=100, bbox_inches='tight')
    plt.show()

print("\n" + "="*70)
print("✅ MODEL TRAINING AND EVALUATION COMPLETE")
print("="*70)